In [2]:
%pip install -r requirements.txt

ERROR: Could not find a version that satisfies the requirement anaconda-anon-usage (from versions: none)

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
ERROR: No matching distribution found for anaconda-anon-usage
Note: you may need to restart the kernel to use updated packages.


In [7]:
import torch
import json
from chunking import SectionAwareChunker, SemanticChunker
from embeddings import MiniLMEmbedding, QwenEmbedding
from vectore_store import FAISSVectorDB, ChromaVectorDB
from ranking_n_retrieval import Retriever
from llm_n_prompt import QwenLLM, PromptTemplate
from rag_pipeline import RAGPipeline
from evaluation import evaluate_rag_pipeline

In [2]:
# ==============================================================
# CONFIGURATION — change these to test different combinations
# ==============================================================
def get_device():
    if torch.cuda.is_available():
        return "cuda"
    elif torch.backends.mps.is_available():
        return "mps"
    else:
        return "cpu"

CHUNKER    = "section"  # "section" or "semantic"
EMBEDDING  = "minilm"   # "minilm"  or  "qwen"
VECTORDB   = "faiss"    # "faiss"   or  "chroma"
RETRIEVAL  = "combo2"   # "combo1"  or  "combo2"
DEVICE     = get_device()
FILEPATHS  = [
    "data/raw/south_asian_corpus.json",
    "data/raw/saved_wikibook_data.json",
    "data/raw/cuisines80.json"
]

In [3]:
def build_embedder(choice):
    if choice == "minilm":
        return MiniLMEmbedding(), 384
    elif choice == "qwen":
        return QwenEmbedding(), 768
    else:
        raise ValueError(f"Unknown embedder: {choice}. Choose 'minilm' or 'qwen'")


def build_vectordb(choice, dim):
    if choice == "faiss":
        return FAISSVectorDB(dim=dim)
    elif choice == "chroma":
        return ChromaVectorDB()
    else:
        raise ValueError(f"Unknown vectordb: {choice}. Choose 'faiss' or 'chroma'")


def build_retriever(choice, vectordb, documents):
    retriever = Retriever(vectordb, documents)
    retriever.active_combo = choice   # store choice on the object
    return retriever


In [4]:
def run_json_input_output(input_file_name, output_file_name):

    # print("\n" + "="*50)
    # print("       CuisineRAG — ChefBot")
    # print("="*50)
    # print(f"Chunker: {CHUNKER}")
    # print(f"  Embedding : {EMBEDDING}")
    # print(f"  VectorDB  : {VECTORDB}")
    # print(f"  Retrieval : {RETRIEVAL}")
    # print(f"  Device    : {DEVICE}")
    # print("="*50 + "\n")

    # --- build components based on config ---
    if CHUNKER == "semantic":
        chunker = SemanticChunker()       # reuses all-MiniLM-L6-v2
    else:
        chunker = SectionAwareChunker()
    embedder, dim = build_embedder(EMBEDDING)
    vectordb      = build_vectordb(VECTORDB, dim)
    prompt_builder = PromptTemplate()
    llm           = QwenLLM(device=DEVICE)

    # --- build pipeline (retriever added after indexing) ---

    pipeline = RAGPipeline(
        chunker        = chunker,
        embedder       = embedder,
        vectordb       = vectordb,
        retriever      = None,
        prompt_builder = prompt_builder,
        llm            = llm
    )

    # --- index the cuisine data ---

    import os

    INDEX_BIN  = "faiss_index.bin"
    DOCS_JSON  = "faiss_docs.json"

    if os.path.exists(INDEX_BIN) and os.path.exists(DOCS_JSON):
        # already indexed — just load from disk
        vectordb.load(INDEX_BIN, DOCS_JSON)
        pipeline.chunks = vectordb.documents
        print("Loaded existing index from disk.")
    else:
        # first run — index and save
        pipeline.index_data(FILEPATHS)
        vectordb.save(INDEX_BIN, DOCS_JSON)

    # --- build retriever with chunks for BM25 ---

    retriever = build_retriever(RETRIEVAL, vectordb, pipeline.chunks)
    pipeline.retriever = retriever

    with open(input_file_name) as input_file:
        query_list=json.load(input_file)['queries']

    print(f"\nProcessing {len(query_list)} queries from {input_file_name}...\n")

    results=[0]*len(query_list)
    for i,query in enumerate(query_list):
        query_id=query["query_id"]
        query_text=query["query"].strip()
        print(f"[{i+1}/{len(query_list)}] {query_text[:60]}...")
        answer,docs=pipeline.query(query_text)
        results[i]={
            "query_id":str(query_id),
            "query":str(query_text),
            "response":str(answer),
            "retrieved_context":[{"doc_id":chunk.metadata.get('doc_id', '?'),"text":chunk.page_content} for chunk in docs]
            }
        final=dict({"results": results})

    with open(output_file_name,'w') as output_json:
        json.dump(final, output_json, indent=2, ensure_ascii=False)

    print(f"\nDone! {len(results)} results saved to {output_file_name}")



Please put your input .json file in the **inputs_and_outputs** folder, this folder will also contain your output after you run the 
**run_json_input_output()** function.

In [5]:
input_file_path = "inputs_and_outputs/input_payload_sample.json" #change as per your file name
output_file_path = "inputs_and_outputs/output_payload_sample.json" #change if needed or leave as is

In [6]:
if __name__ == "__main__":
    run_json_input_output(input_file_path, output_file_path)

Loading MiniLM embedding model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7377.65it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Initializing FAISS...
Loading LLM...


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 4206.37it/s]


Loaded 327 docs from data/raw/south_asian_corpus.json
Loaded 716 docs from data/raw/saved_wikibook_data.json
Loaded 6 docs from data/raw/cuisines80.json
Indexing 1,049 documents total...
Finished! Total chunks: 9888
FAISS saved → faiss_index.bin, faiss_docs.json
Loading cross-encoder reranker...


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 5192.07it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Processing 14 queries from inputs_and_outputs/input_payload_sample.json...

[1/14] What is Dosa and where does it originate from?...
[2/14] What is Samosas and where does it originate from?...
[3/14] What is Chicken Tikka Masala and where is it from?...
[4/14] What is a biryani?...
[5/14] What is anglo-Indian food?...
[6/14] What is Pongal?...
[7/14] What is Garam Masala?...
[8/14] What is upma?...
[9/14] What is ghevar usually served with?...
[10/14] What is Nihari?...
[11/14] What is Kashmiri Pink Tea?...
[12/14] What is the main fish used in Bengali cuisine?...
[13/14] What are Sri Lankan Hoppers?...
[14/14] What is Chana Dal Halwa?...

Done! 14 results saved to inputs_and_outputs/output_payload_sample.json


### EVALUATION PIPELINE

In [ ]:
# Replace these filenames with the actual paths to your JSON files
eval_df = evaluate_rag_pipeline(
        output_path='inputs_and_outputs/output_payload_sample.json', 
        benchmark_path='data/benchmark_dataset.json'
    )

  RAG PIPELINE EVALUATION

[Setup] Loading models …


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5859.95it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 389/389 [00:00<00:00, 12250.88it/s]
RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTE

[Setup] Done.



Evaluating queries: 100%|███████████████████████| 14/14 [00:03<00:00,  4.14it/s]


  RESULTS SUMMARY

── GENERATION METRICS ──────────────────────────────────────
  BLEU (nltk)          : 0.0980
  ROUGE-1 F1           : 0.3184
  ROUGE-1 Precision    : 0.2387
  ROUGE-1 Recall       : 0.5938
  SacreBLEU            : 9.68
  ChrF3++              : 40.02
  BERTScore F1         : 0.8923

── RETRIEVAL METRICS ───────────────────────────────────────
  Keyword Recall       : 75.35%
  Semantic Similarity  : 0.7423
  Precision            : 0.0000
  Recall               : 0.0000
  MRR                  : 0.0000

